In [46]:
%%time

import pandas as pd
import numpy as np
import random

import matplotlib.pyplot as plt
import seaborn as sns

import logging
import warnings

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("lucky_winner.log", mode='a', encoding='utf-8')  # файл
    ],
    force=True
)

warnings.filterwarnings('ignore')

df = pd.read_csv('res_with_reg.csv')

CPU times: user 3.7 s, sys: 381 ms, total: 4.08 s
Wall time: 4.17 s


In [47]:
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d', errors='coerce')

In [48]:
df['Net Cash EUR'] = df['dep_sum'] - df['with_sum']
df['VIP level'] = 3

df = df.rename(columns={
    'user_id': 'Account ID',
    'date': 'Date',
    'ggr': 'GGR'
})

df.head()

,Account ID,Date,country,registration_date,bet_sum,GGR,dep_sum,with_sum,Net Cash EUR,VIP level
0,1000000159,2025-01-07,Польша,2024-09-25 20:01:53,23.68,-15.63,23.68,0.00,23.68,3
1,1000000159,2025-01-09,Польша,2024-09-25 20:01:53,0.00,0.00,0.00,39.20,-39.20,3
2,1000000159,2025-01-10,Польша,2024-09-25 20:01:53,0.00,0.00,23.45,0.00,23.45,3
3,1000000159,2025-01-11,Польша,2024-09-25 20:01:53,0.00,0.00,35.12,56.88,-21.76,3
4,1000000159,2025-01-12,Польша,2024-09-25 20:01:53,11.46,11.46,11.71,24.89,-13.18,3


In [49]:
df['m'] = pd.to_datetime(df['registration_date']).dt.month

In [50]:
data = df.copy()

data.columns

Index(['Account ID', 'Date', 'country', 'registration_date', 'bet_sum', 'GGR',
       'dep_sum', 'with_sum', 'Net Cash EUR', 'VIP level', 'm'],
      dtype='object')

In [51]:
class LuckyWinner:
    '''
    Class for selecting lucky winners based on given criteria.

    Attributes:
    - PRIZE: The prize amount for each winner.
    - STD: The standard deviation used in upper border detection.
    - WINNERS_COUNT: A dictionary mapping group names to the number of winners for each group.
    - GROUP_WEIGHTS: A dictionary mapping group names to their corresponding weights.
    - WEIGHTS: A dictionary mapping different metrics to their weights.
    - MAX_CONSECUTIVE_WINS: The maximum allowed consecutive wins for a user.
    - MIN_CHANCE: The minimum chance for winning (0%).
    - MAX_CHANCE: The maximum chance for winning (10%).

    Methods:
    - __init__(self, data, items=None, k=None, prev_winners=None, consecutive_winners=None, group_name=None, group=None, weights=WEIGHTS, group_weights=GROUP_WEIGHTS): Initializes the LuckyWinner object with given parameters.
    - users(self, df): Removes users with high GGR (Gross Gaming Revenue) metric per day and selects lucky winners based on specified criteria.
    - classification(self, df): Classifies rows of a DataFrame into different groups based on the values in several columns.
    - run_winner_selection(self, df, max_consecutive_wins, min_chance, max_chance): Runs the winner selection process for each date in the given DataFrame.
    - weighted_random_sample(self, items, k, prev_winners, consecutive_winners, group_name): Selects a weighted random sample of winners based on given criteria.
    - select_winners(self, group, prev_winners, consecutive_winners, min_chance, max_chance): Selects winners from a group based on specified criteria.
    - reward_calculation(self, group, prize, weights): Calculates the reward for each user in a group based on specified metrics and weights.
    - reward_adjustment(self, df, group_weights): Adjusts the rewards based on group weights.
    '''
    
    PRIZE = 500 # ROI Effect - very strong
    STD = 0.5
    
    WINNERS_COUNT = {
        'Group 1': 9, # ROI Effect - strong
        'Group 2': 10, # ROI Effect - low/med
        'Group 3': 4, # ROI Effect - low/med
        'Group 4': 2, # ROI Effect - low/med
        'Other': 0
        }

    GROUP_WEIGHTS = {
        'Group 1': 0.5,
        'Group 2': 1,
        'Group 3': 2,
        'Group 4': 2
        }

    WEIGHTS = {
        'sqrt': 0.5, 
        'ln': 0.3, 
        'Z-score': 0.2
        }
    
    MAX_CONSECUTIVE_WINS = 4  # Maximum allowed consecutive wins
    MIN_CHANCE = 0.0  # Minimum chance for winning (0%)
    MAX_CHANCE = 0.1  # Maximum chance for winning (10%)
    
    
    
    def __init__ (self, data,
                  items=None, k=None, prev_winners=None, consecutive_winners=None, group_name=None,
                  group=None, weights=WEIGHTS, group_weights=GROUP_WEIGHTS):
        '''
        Initializes the LuckyWinner object.

        Parameters:
        - data: The data used for the winner selection process.
        - items: A list of items to select winners from.
        - k: The number of winners to select.
        - prev_winners: A list of previous winners.
        - consecutive_winners: A dictionary mapping user IDs to the number of consecutive wins.
        - group_name: The name of the group.
        - group: The group information.
        - weights: A dictionary mapping different metrics to their weights.
        - group_weights: A dictionary mapping group names to their corresponding weights.
        '''
        
        self.data = data
        self.items = items
        self.k = k
        self.prev_winners = prev_winners
        self.consecutive_winners = consecutive_winners
        self.group_name = group_name
        self.group = group
        self.weights = weights
        self.group_weights = group_weights
        
        self.std = LuckyWinner.STD
        self.prize = LuckyWinner.PRIZE
        self.min_chance = LuckyWinner.MIN_CHANCE
        self.max_chance = LuckyWinner.MAX_CHANCE
        self.max_consecutive_wins = LuckyWinner.MAX_CONSECUTIVE_WINS

        
      
    def users(self, df):
        '''
        Removes users with high GGR (Gross Gaming Revenue) metric per day and selects lucky winners based on specified criteria.

        Parameters:
        - df: Input dataframe. Expected to have at least the columns 'date', 'account_id', and 'GGR'.

        Returns:
        - result: Output dataframe, containing the account_id, date, and sum of GGR for each user and day,
                  excluding users with a high GGR metric. The 'date' column added in the final output corresponds
                  to the date for which the metrics were calculated.
        '''

        final_dataframes = []

        # Taking all availiable dates
        dates = np.sort(df['Date'].unique())

        # Iterating through each date
        for date in dates:

            # Filtering out dataframe by particular date
            daily_df = df[df['Date'] == date].copy()

            # Remove users with GGR == 0
            daily_df = daily_df[daily_df['GGR'] != 0]
            
            # if daily_df.empty:
                # continue
            
            # Logging
            try:
                if daily_df.empty:
                    logging.info(f"Skipped {date}: all users have GGR = 0")
                    continue
                    
                # Creating 2 groups to identify outliers (2.5% from each side)
                daily_df['lower'] = np.where(daily_df['GGR'] < np.quantile(daily_df['GGR'], 0.025), 1, 0)
                daily_df['upper'] = np.where(daily_df['GGR'] > np.quantile(daily_df['GGR'], 0.975), 1, 0)
        
            except Exception as e:
                logging.error(f"Error processing {date}: {type(e).__name__} – {e}")
                continue
            

            # Removing Outliers (users in groups lower and upper)
            daily_outliers = daily_df[(daily_df['lower']==0) & (daily_df['upper']==0)]

            # if daily_outliers.empty:
                # continue

            # Calculating mean, sts, and border for each day
            outliers_mean = daily_outliers['GGR'].mean()
            outliers_std = daily_outliers['GGR'].std()
            outliers_border = outliers_mean + outliers_std * self.std

            # Removing users with daily total GGR value greater than the border value
            daily_final = daily_df[daily_df['GGR'] < outliers_border]
            daily_final = daily_final.drop('upper', axis=1)
            daily_final['Date'] = date
            daily_final['mean'] = outliers_mean

            # Append modified daily data to the dataframe
            final_dataframes.append(daily_final)

        # Concatenating all results into one dataframe
        result = pd.concat(final_dataframes)
        result = result[['Date', 'Account ID', 'GGR', 'Net Cash EUR', 'VIP level', 'mean', 'lower']]
        
        # Apply the classification method to create the 'group' column
        result['group'] = result.apply(self.classification, axis=1)
        
        df_winners = self.run_winner_selection(result, self.max_consecutive_wins, self.min_chance, self.max_chance)
        
        winners = df_winners.groupby('Date', group_keys=True).apply(lambda group: self.reward_calculation(group, LuckyWinner.PRIZE, LuckyWinner.WEIGHTS)).reset_index(drop=True)
        
        adjusted_winners = self.reward_adjustment(winners, LuckyWinner.GROUP_WEIGHTS, LuckyWinner.PRIZE)
        adjusted_winners = adjusted_winners[['Date', 'Account ID', 'GGR', 'Net Cash EUR', 'VIP level', 'group', 'reward', 'adjusted_reward', 'final_reward']]
        return adjusted_winners
    
    
    
    def classification(self, df):
        '''
        Classifies rows of a DataFrame into different groups based on the values in several columns.

        Parameters:
        - df: The DataFrame to classify.

        Returns:
        - group: The group name assigned to the row.
        '''

        # Creating conditions for each group
        if df['GGR'] > df['mean']:
            return 'Group 1'
        elif df['GGR'] >= 0: 
            return 'Group 2'
        elif (df['GGR'] < 0) & (df['lower'] != 1):
            return 'Group 3'
        elif (df['GGR'] < 0) & (df['lower'] == 1) & (df['VIP level'] != 1):
            return 'Group 4'
        else:
            return 'Other'
        
        
        
    def run_winner_selection(self, df, max_consecutive_wins, min_chance, max_chance):
        '''
        Runs the winner selection process for each date in the given DataFrame.

        Parameters:
        - df: The DataFrame containing the data for winner selection.
        - max_consecutive_wins: The maximum allowed consecutive wins for a user.
        - min_chance: The minimum chance for winning (0%).
        - max_chance: The maximum chance for winning (10%).

        Returns:
        - df_winners: The DataFrame containing the selected winners.
        '''
        
        df_winners = pd.DataFrame()
        consecutive_winners = {id: 0 for id in df['Account ID'].unique()}
        prev_winners = []
        
        # Iterate over unique dates in the dataframe
        for date in sorted(df['Date'].unique()):
            # Filter the dataframe for the current date
            daily_df = df[df['Date'] == date]
            # Exclude outliers from the winners selection process
            daily_df_filtered = daily_df[daily_df['group'] != 'Other']
            # Group the filtered data by date and group and apply the winners selection function
            daily_winners = daily_df_filtered.groupby(['Date', 'group']).apply(lambda group: self.select_winners(group, prev_winners, consecutive_winners, min_chance, max_chance)).reset_index(drop=True)
            winners_df = daily_winners[['Date', 'Account ID', 'GGR', 'Net Cash EUR', 'group', 'VIP level']]
            
            # Print winners for the current date if any
            if not winners_df.empty:
                # logging.info(f"\nWinners for {date}:")
                # logging.info(winners_df)
                prev_winners = winners_df['Account ID'].tolist()
                df_winners = pd.concat([df_winners, winners_df])
                
        return df_winners
    
    
    
    def weighted_random_sample(self, items, k, prev_winners, consecutive_winners, group_name):
        '''
        Selects a weighted random sample of winners based on given criteria.

        Parameters:
        - items: The list of items to select winners from.
        - k: The number of winners to select.
        - prev_winners: A list of previous winners.
        - consecutive_winners: A dictionary mapping user IDs to the number of consecutive wins.
        - group_name: The name of the group.

        Returns:
        - winners: The list of selected winners.
        '''
        
        weights = []
        for item in items:
            weight = 1.0
            if item in prev_winners and consecutive_winners[item] < self.max_consecutive_wins:
                weight += random.uniform(0.0, self.max_chance)
            elif item not in consecutive_winners or group_name == 'Outliers':
                weight += random.uniform(0.0, self.max_chance)
            weights.append(weight)
        weights /= np.sum(weights)  # Normalize weights to sum up to 1
        winners_indices = np.random.choice(len(items), size=k, replace=False, p=weights)
        winners = [items[idx] for idx in winners_indices]
        return winners
    
    
    
    def select_winners(self, group, prev_winners, consecutive_winners, min_chance, max_chance):
        '''
        Selects winners from a group based on specified criteria.

        Parameters:
        - group: The group to select winners from.
        - prev_winners: A list of previous winners.
        - consecutive_winners: A dictionary mapping user IDs to the number of consecutive wins.
        - min_chance: The minimum chance for winning (0%).
        - max_chance: The maximum chance for winning (10%).

        Returns:
        - group: The DataFrame containing the selected winners from the group.
        '''
        
        group_name = group.name[1]
        account_ids = group['Account ID'].tolist()  # Please, ensure that 'Account ID' is the correct column name in your DataFrame.
        # winners_count_group needs to be calculated or passed as an argument
        winners_count_group = min(self.WINNERS_COUNT[group_name], len(account_ids))
        winners = self.weighted_random_sample(account_ids, winners_count_group, prev_winners, consecutive_winners, group_name)
        for winner_id in winners:
            if winner_id in prev_winners:
                # Determine the win chance for a previous winner
                win_chance = random.uniform(min_chance, max_chance)
                if random.random() < win_chance and consecutive_winners[winner_id] < self.max_consecutive_wins:
                    consecutive_winners[winner_id] += 1
                    # logging.info(f"On {group.name[0]}, User {winner_id} is a winner {consecutive_winners[winner_id]} days in a row!")
            elif winner_id not in consecutive_winners or group_name == 'Outliers':
                # Determine the win chance for an inactive winner or an outlier winner
                win_chance = random.uniform(min_chance, max_chance)
                # if random.random() < win_chance:
                #     if winner_id not in consecutive_winners:
                #         logging.info(f"On {group.name[0]}, User {winner_id} is an inactive winner!")
                #     elif group_name == 'Outliers':
                #         logging.info(f"On {group.name[0]}, User {winner_id} from Outliers group is a winner!")
        return group[group['Account ID'].isin(winners)]
    
    
    
    def reward_calculation(self, group, prize, weights):
        '''
        Calculates the reward for each user in a group based on specified metrics and weights.

        Parameters:
        - group: The group to calculate rewards for.
        - prize: The prize amount for each winner.
        - weights: A dictionary mapping metrics to their weights.

        Returns:
        - group: The group DataFrame with calculated rewards.
        '''
    
        group['sqrt'] = np.sqrt(abs(group['GGR']))
        group['ln'] = np.log(abs(group['GGR']) + 1e-6)
        group['Z-score'] = abs((group['GGR'] - group['GGR'].mean()) / group['GGR'].std())

        for metric in ['Z-score', 'sqrt', 'ln']:
            group[f'{metric}-val'] = group[metric] / group[metric].sum() * prize
        
        group['reward'] = abs(round(sum(group[col+'-val'] * weights[col] for col in weights.keys()), 2))
        reward_sum = group['reward'].sum()

        return group
    
    
    
    def reward_adjustment(self, df, group_weights, prize):
        '''
        Adjusts the rewards based on group weights.

        Parameters:
        - df: The DataFrame containing the rewards.
        - group_weights: A dictionary mapping group names to their corresponding weights.

        Returns:
        - df: The DataFrame with adjusted rewards.
        '''
        
        df['adjusted_reward'] = df['reward'].copy()  # Create a new column with original rewards

        for group, weight in group_weights.items():
            # Apply the weight only to the rows of the specific group
            df.loc[df['group'] == group, 'adjusted_reward'] *= weight

        for date in np.sort(df['Date'].unique()):
            # Create a mask for the current date
            date_mask = df['Date'] == date

            adjusted_reward_sum = df.loc[date_mask, 'adjusted_reward'].sum()
            reward_sum = df.loc[date_mask, 'reward'].sum()

            # Normalize the 'adjusted_reward' so they sum up to prize
            df.loc[date_mask, 'adjusted_reward'] = prize * df.loc[date_mask, 'adjusted_reward'] / adjusted_reward_sum

            # Ensure the adjusted reward is not lower than the original reward
            df.loc[date_mask, 'adjusted_reward'] = df.loc[date_mask, ['adjusted_reward', 'reward']].max(axis=1)

            df.loc[date_mask, 'final_reward'] = df.loc[date_mask, 'adjusted_reward'] - ((df.loc[date_mask, 'reward'] / reward_sum) * (adjusted_reward_sum - prize))

            # Normalize 'final_reward' so the total per date sums up to prize
            df.loc[date_mask, 'final_reward'] = round(prize * df.loc[date_mask, 'final_reward'] / df.loc[date_mask, 'final_reward'].sum(), 2)

        return df

In [60]:
%%time

lw = LuckyWinner(data)
# result = lw.users(data)  
result = lw.users(data[data['Date'] == '2025-08-01']) 

# date = '2025-08-01' 

new_r = result[result['Date'] == date]

display(new_r[['Date', 'Account ID', 'GGR', 'Net Cash EUR', 'VIP level', 'group', 'final_reward']])
print(f"reward: {new_r['reward'].sum()}")
print(f"adjusted reward: {new_r['adjusted_reward'].sum()}")
print(f"final reward: {new_r['final_reward'].sum()}")
print(f"net cash: {new_r['Net Cash EUR'].sum()}")

,Date,Account ID,GGR,Net Cash EUR,VIP level,group,final_reward
0,2025-08-01,1163601339,9.83,11.71,3,Group 1,15.90
1,2025-08-01,1215732013,10.52,-13.52,3,Group 1,16.39
2,2025-08-01,1290954893,8.22,8.25,3,Group 1,14.64
3,2025-08-01,1338199005,9.93,10.01,3,Group 1,15.97
4,2025-08-01,1338293345,10.00,9.99,3,Group 1,16.02
5,2025-08-01,452756559,10.18,16.90,3,Group 1,16.15
6,2025-08-01,949029181,9.54,10.09,3,Group 1,15.68
7,2025-08-01,951640333,12.95,12.94,3,Group 1,17.98
8,2025-08-01,989534407,13.52,13.52,3,Group 1,18.33
9,2025-08-01,1257024007,3.94,4.00,3,Group 2,10.16


reward: 501.78999999999996
adjusted reward: 635.5788558132072
final reward: 499.99
net cash: 197.8
CPU times: user 151 ms, sys: 8.12 ms, total: 159 ms
Wall time: 155 ms


In [120]:
new_r.to_clipboard()